# Web-Gold-40K — causal smoke and mini training

This notebook tests the 39,215-row structured dataset on Kaggle. It intentionally supports only two development stages:

- **smoke**: 16 training rows; verifies causal inputs, shapes, finite loss, backward pass, and a real optimizer update.
- **mini**: 5,000 stratified training rows and 500 validation rows for 5 epochs; verifies learning, checkpoint round-trip, and saves a per-epoch CSV.

Causal routing keeps every allowed input: `state_before` drives action/bbox/confidence-before, while `state_before + state_after` drives outcome/failure/recovery/memory. Both paths share the same VLM and adapter. The test split is never read by either stage.

This is not the headline/full-training notebook. Run smoke first, inspect its report, then change `STAGE` to `mini` and restart the kernel.

In [1]:
# 1. Pull the modular training code and record the environment.
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)],
        check=True,
    )

requirements = [
    'transformers>=4.49,<5',
    'peft>=0.14,<1',
    'bitsandbytes>=0.45,<1',
    'accelerate>=1,<2',
    'scikit-learn>=1.4,<2',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)

packages = ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']
environment = {name: metadata.version(name) for name in packages}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
environment_path = Path('/kaggle/working/gold_stage_environment.json')
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))
print('Environment saved:', environment_path)

Cloning into '/kaggle/working/webagent'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
{
  "torch": "2.10.0+cu128",
  "transformers": "4.57.6",
  "peft": "0.19.1",
  "bitsandbytes": "0.49.2",
  "accelerate": "1.14.0",
  "scikit-learn": "1.9.0",
  "python": "3.12.13",
  "git_commit": "729af831df3ba3c597d6de54749888982cffcba7"
}
Environment saved: /kaggle/working/gold_stage_environment.json


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
# 2. Locate the attached 39,215-row dataset without downloading or extracting it.
EXPECTED_DATASET_ROOT = Path(
    '/kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k'
)
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')

def contains_splits(path: Path) -> bool:
    return path.is_dir() and all((path / name).is_file() for name in SPLIT_FILES)

def find_split_root() -> Path:
    if contains_splits(EXPECTED_DATASET_ROOT):
        return EXPECTED_DATASET_ROOT
    candidates = []
    if ATTACHED_ROOT.is_dir():
        for current, _, files in os.walk(ATTACHED_ROOT, followlinks=True):
            if set(SPLIT_FILES).issubset(files):
                candidates.append(Path(current))
    if not candidates:
        raise FileNotFoundError(
            'Structured split folder not found. Attach kiyasmahmud/web-gold-40k; '
            'smoke/mini training requires the mounted Kaggle folder, not a local download.'
        )
    return sorted(candidates, key=lambda path: (len(path.parts), str(path)))[0]

DATA_ROOT = find_split_root().resolve()
print('DATA_ROOT =', DATA_ROOT)
print('Split files:', [name for name in SPLIT_FILES if (DATA_ROOT / name).is_file()])
print('Dataset stays read-only under /kaggle/input.')

DATA_ROOT = /kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k
Split files: ['split_train.json', 'split_val.json', 'split_test.json']
Dataset stays read-only under /kaggle/input.


In [3]:
# 3. Choose exactly one development stage. Run smoke before mini.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed

STAGE = 'mini'  # 'smoke' or 'mini'
SEED = 42
SMOKE_ROWS = 16
MINI_TRAIN_ROWS = 5_000
MINI_VAL_ROWS = 500
MINI_EPOCHS = 5

assert STAGE in {'smoke', 'mini'}
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)

cfg = load_config('configs/backbones/qwen2vl_2b_gold.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['causal_routing'] = True
cfg['data']['use_state_after'] = True
cfg['data']['use_visual_diff_text'] = False
cfg['data']['num_workers'] = 0 if STAGE == 'smoke' else 4

print('GPU:', torch.cuda.get_device_name(0))
print('STAGE:', STAGE)
print('pre-action  -> state_before + task/domain -> action, bbox, confidence-before')
print('post-action -> before + after + task/domain -> outcome, failure, recovery, memory')
print('early-stop metric:', cfg['train']['early_stop_metric'])

GPU: Tesla T4
STAGE: mini
pre-action  -> state_before + task/domain -> action, bbox, confidence-before
post-action -> before + after + task/domain -> outcome, failure, recovery, memory
early-stop metric: outcome_mcc


In [4]:
# 4. Inspect one causal batch before allocating the model.
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split
from web_agent.train.gold_stages import build_processor

processor = build_processor(cfg)
train_records = load_gold_split(cfg, 'train')
val_records = load_gold_split(cfg, 'val')
assert len(train_records) == 23_499, f'Unexpected train rows: {len(train_records)}'
assert len(val_records) == 7_861, f'Unexpected validation rows: {len(val_records)}'

inspection_loader = build_gold_dataloader(
    cfg, 'train', processor, records=train_records, limit=16, batch_size=4,
    shuffle=False, num_workers=0, seed=SEED, smoke=True,
)
inspection_batch = next(iter(inspection_loader))
required_streams = {
    'pre_input_ids', 'pre_pixel_values', 'pre_image_grid_thw',
    'post_input_ids', 'post_pixel_values', 'post_image_grid_thw',
}
assert required_streams.issubset(inspection_batch)
assert 'input_ids' not in inspection_batch, 'Unrouted shared stream would leak state_after.'

print('train rows:', len(train_records), '| validation rows:', len(val_records))
for key, value in inspection_batch.items():
    if torch.is_tensor(value):
        print(f'{key:28} shape={tuple(value.shape)} dtype={value.dtype}')
print('Pre images in batch:', inspection_batch['pre_image_grid_thw'].shape[0])
print('Post images in batch:', inspection_batch['post_image_grid_thw'].shape[0])

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

train rows: 23499 | validation rows: 7861
bbox                         shape=(4, 4) dtype=torch.float32
bbox_mask                    shape=(4, 1) dtype=torch.float32
borrowed                     shape=(4, 1) dtype=torch.float32
label_outcome                shape=(4,) dtype=torch.int64
label_failtype               shape=(4,) dtype=torch.int64
label_action                 shape=(4,) dtype=torch.int64
label_recovery               shape=(4,) dtype=torch.int64
label_memory                 shape=(4, 1) dtype=torch.float32
label_confidence             shape=(4, 1) dtype=torch.float32
label_recovery_success       shape=(4, 1) dtype=torch.float32
pre_input_ids                shape=(4, 306) dtype=torch.int64
pre_attention_mask           shape=(4, 306) dtype=torch.int64
pre_pixel_values             shape=(4032, 1176) dtype=torch.float32
pre_image_grid_thw           shape=(4, 3) dtype=torch.int64
post_input_ids               shape=(4, 563) dtype=torch.int64
post_attention_mask          shape=(4, 5

In [5]:
# 5. Run the selected stage. Mini uses validation only and never opens split_test.json.
from web_agent.train.gold_stages import run_gold_mini, run_gold_smoke
from web_agent.utils.results import save_mini_result_csv

if STAGE == 'smoke':
    stage_report = run_gold_smoke(
        cfg, processor=processor, rows=SMOKE_ROWS, seed=SEED,
    )
else:
    stage_report = run_gold_mini(
        cfg, processor=processor, train_rows=MINI_TRAIN_ROWS,
        val_rows=MINI_VAL_ROWS, epochs=MINI_EPOCHS, seed=SEED,
    )

report_path = Path(f'/kaggle/working/gold_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = None
if STAGE == 'mini':
    result_csv_path = save_mini_result_csv(
        stage_report, '/kaggle/working/gold_mini_result.csv',
    )
print(json.dumps(stage_report, indent=2))
print('Report saved:', report_path)
if result_csv_path is not None:
    print('CSV saved:', result_csv_path)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290
train rows: 23499
action weights: [1.02, 1.01, 1.01, 1.01, 0.88, 1.1]
failure weights: [0.57, 0.98, 0.9, 8.56]
outcome weights: [1.28, 1.0]
recovery pos_weight: 1.92 (attempted success=1888, failure=3630)
epoch 0 | step 50/157 | loss 1.003 | elapsed 34.4min | ETA 73.5min
epoch 0 | step 100/157 | loss 0.742 | elapsed 68.8min | ETA 39.2min
epoch 0 | step 150/157 | loss 0.955 | elapsed 103.4min | ETA 4.8min
epoch 0: train_loss=0.9242  failure_f1=0.7289  failure_macro_f1=0.6862  outcome_bal_acc=0.6886  outcome_mcc=0.3738  success_recall=0.6683  failtype_acc=0.3980  failtype_macro_f1=0.3393  action_acc=0.3540  action_macro_f1=0.2688  recovery_acc=0.7500  recovery_outcome_acc=0.7600  recovery_outcome_mcc=0.3532  memory_acc=0.7340  outcome_ece=0.0235  confidence_mae=0.1604  confidence_success_ece=0.1037  bbox_mae=0.1786
epoch 1 | step 50/157 | loss 1.145 | elapsed 34.6min | ETA 74.0min
epoch 1 | step 100/157 | los

In [6]:
# 6. Enforce the stage gate and state the next permitted action.
assert stage_report['status'] == 'PASS'

if STAGE == 'smoke':
    assert stage_report['dataset_rows'] == 16
    assert stage_report['processed_rows'] == 16
    assert stage_report['parameter_changed'] is True
    print('SMOKE PASSED. Restart the kernel, change STAGE to mini, then Run All.')
else:
    assert stage_report['train_rows'] == 5_000
    assert stage_report['val_rows'] == 500
    assert stage_report['test_rows_read'] == 0
    assert stage_report['loss_decreased'] is True
    assert stage_report['checkpoint_roundtrip'] is True
    assert len(stage_report['history']) == MINI_EPOCHS
    assert result_csv_path is not None and result_csv_path.is_file()
    print('MINI PASSED. Preserve the report and checkpoint; do not inspect test yet.')
    print('Download the per-epoch CSV:', result_csv_path)
    print('Full/headline training remains blocked until manual review and full image hashing pass.')

MINI PASSED. Preserve the report and checkpoint; do not inspect test yet.
Download the per-epoch CSV: /kaggle/working/gold_mini_result.csv
Full/headline training remains blocked until manual review and full image hashing pass.


## Interpretation rules

- A smoke pass proves only that the causal pipeline executes correctly. It is not a model result.
- A mini pass requires decreasing training loss, a positive validation `outcome_mcc`, and an exact checkpoint prediction round-trip.
- Development compares models using validation metrics. The test split is reserved for the final frozen evaluation.
- `state_after` remains essential for post-action failure and recovery learning; it is blocked only from pre-action action/bbox/confidence heads.
- Do not start headline training until review decisions and full SHA-256/dHash image-overlap gates are complete.